# 02 — Features: Skills Matrix and Symmetric Adjacency

**Project H20 — Succession-Planning Graph Recommender.** We build the per-employee skill matrix S (normalised) and the row-normalised symmetric adjacency A. Stacked horizontally as `[S | A]` they feed the TruncatedSVD spectral embedding in notebook 03.

In [ ]:
import sys, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

sns.set_theme(style='whitegrid')
sys.path.insert(0, '../src')
from succession_graph.features import skills_matrix, adjacency_matrix
df = pd.read_parquet('../data/processed/employee_attrs.parquet')
df['skills'] = df['skills'].apply(np.asarray)
with open('../data/processed/org_graph.gpickle', 'rb') as fh:
    G = pickle.load(fh)
len(df), G.number_of_nodes()

## 1. Skills matrix S

In [ ]:
S = skills_matrix(df)
print(f'S shape: {S.shape}')
print(f'row L2 norms — min={np.linalg.norm(S, axis=1).min():.3f}  max={np.linalg.norm(S, axis=1).max():.3f}')

## 2. Adjacency A

In [ ]:
emp_index = list(df['emp_id'].values)
A = adjacency_matrix(G, emp_index)
print(f'A shape: {A.shape}')
print(f'row sums  min={A.sum(axis=1).min():.3f}  max={A.sum(axis=1).max():.3f}')
print(f'symmetric: {np.allclose(A, A.T) is False} (row-normalised, so unsurprising)')

## 3. Visualise A as a sparsity heatmap (subset)

In [ ]:
sub = A[:200, :200]
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(sub > 0, cmap='binary', aspect='auto')
ax.set_title('Adjacency A (first 200 employees) — non-zero pattern')
plt.tight_layout(); plt.show()

## 4. Concatenate `[S | A]`

In [ ]:
M = np.hstack([S, A])
print(f'concatenated matrix M: {M.shape}')
print(f'fraction of zeros in skills block: {(S == 0).mean():.3f}')
print(f'fraction of zeros in adjacency block: {(A == 0).mean():.3f}')

## 5. Skill-feature mean per role

In [ ]:
S_norm = S
fig, ax = plt.subplots(figsize=(13, 5))
rows = []
for role, sub in df.groupby('role'):
    rows.append((role, S_norm[sub.index].mean(axis=0)))
M_role = np.vstack([m for _, m in rows])
sns.heatmap(M_role, ax=ax, cmap='RdBu_r', center=0,
            yticklabels=[r for r, _ in rows], xticklabels=False)
ax.set_title('Mean normalised skill vector per role'); plt.tight_layout(); plt.show()

## 6. Adjacency degree per role

In [ ]:
ud = G.to_undirected()
degs = pd.Series([ud.degree(e) for e in df['emp_id']], index=df['emp_id']).rename('deg').to_frame()
degs = degs.join(df.set_index('emp_id')[['role', 'level']])
print(degs.groupby('role')['deg'].agg(['mean', 'median', 'max']).round(2))
fig, ax = plt.subplots(figsize=(11, 4))
sns.boxplot(data=degs.reset_index(), x='role', y='deg', ax=ax, color='#1f77b4', showfliers=False)
ax.set_title('Undirected degree per role')
plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## 7. Single-source shortest path examples

In [ ]:
ud = G.to_undirected()
ceo = df.loc[df['role'] == 'CEO', 'emp_id'].iloc[0]
spd = nx.single_source_shortest_path_length(ud, ceo, cutoff=6)
depth = pd.Series(spd).rename('depth').to_frame().join(df.set_index('emp_id')['role'])
print(depth.groupby('depth')['role'].count())

## 8. Save the matrices

In [ ]:
import joblib
out = Path('../models'); out.mkdir(exist_ok=True)
joblib.dump(dict(S=S, A=A, M=M, emp_index=emp_index), out / 'feature_matrices.joblib')
print(f'wrote feature matrices -> {out / "feature_matrices.joblib"}')

## 9. Implications
- Skills give a strong per-role signature (heatmap + role-centroid cosine in nb 01).
- Adjacency is sparse but well-defined; the row-normalisation keeps the SVD on a sane scale.
- `[S | A]` is the right input matrix for the spectral stand-in (TruncatedSVD) used in notebook 03 — same intuition the GNN replacement would learn.